In [1]:
import time
import serial
import serial.tools.list_ports
import numpy as np
import pandas as pd
from scipy.signal import find_peaks
from bokeh.io import curdoc, output_notebook, show
from bokeh.layouts import column, row
from bokeh.models import ColumnDataSource, Button, Div, TextInput
from bokeh.plotting import figure

# --- 1. USER PROVIDED CONNECTION UTILITIES ---

def find_arduino(port=None):
    """Get the name of the port that is connected to Arduino."""
    if port is None:
        ports = serial.tools.list_ports.comports()
        for p in ports:
            if p.manufacturer is not None and "Arduino" in p.manufacturer:
                port = p.device
    return port

def handshake_arduino(
    arduino, sleep_time=1, print_handshake_message=False, handshake_code=0):
    """Make sure connection is established by sending
    and receiving bytes."""
    arduino.close()
    arduino.open()
    time.sleep(sleep_time)
    timeout = arduino.timeout
    arduino.timeout = 2
    _ = arduino.read_all()
    arduino.write(bytes([handshake_code]))
    handshake_message = arduino.read_until()
    arduino.write(bytes([handshake_code]))
    handshake_message = arduino.read_until()
    if print_handshake_message:
        print("Handshake message: " + handshake_message.decode())
    arduino.timeout = timeout

# --- 2. BOKEH APP GLOBALS ---

SPO2_A = 110
SPO2_B = 25

# SMOOTHING FACTOR (0.0 to 1.0)
# Lower = Smoother lines, but more lag.
# Higher = Closer to raw data.
# 0.3 is a good balance for a 100Hz signal.
SMOOTHING_ALPHA = 0.1 

app_state = {
    'serial_connection': None,
    'recorded_data': [], 
    'periodic_callback': None,
    'arduino_port': find_arduino(),
    'streaming': False,
    # State variables for the Low Pass Filter (EMA)
    'last_red': None,
    'last_ir': None
}

# --- 3. SIGNAL PROCESSING LOGIC ---

def process_signal(df, window_size=400):
    """
    Analyzes signal using scipy.signal.find_peaks for robust BPM detection.
    Note: Since we are now smoothing the data BEFORE plotting, 
    the input 'df' here is already smoothed!
    """
    if len(df) < window_size:
        return 0, 0, 0, 0

    # Get recent data window
    recent = df.tail(window_size).copy()
    
    # We can skip extra rolling average here if the plot smoothing is sufficient,
    # but keeping a small roll helps robustness against any remaining artifacts.
    ir = recent['ir'].values
    red = recent['red'].values
    timestamps = recent['timestamp'].values

    # --- HEART RATE (SCIPY FIND_PEAKS) ---
    dynamic_prominence = np.std(ir) * 0.5 
    
    # Distance=12 samples (~240ms) prevents counting double peaks
    peaks, _ = find_peaks(ir, distance=12, prominence=dynamic_prominence)
    
    bpm = 0
    mean_ipi = 0
    sdnn = 0

    if len(peaks) > 1:
        peak_times = timestamps[peaks]
        intervals = np.diff(peak_times) 
        
        valid_intervals = intervals[(intervals > 300) & (intervals < 2000)]
        
        if len(valid_intervals) > 0:
            mean_ipi = np.mean(valid_intervals)
            sdnn = np.std(valid_intervals)
            bpm = 60000.0 / mean_ipi

    # --- SpO2 CALCULATION ---
    dc_red = np.mean(red)
    dc_ir = np.mean(ir)
    ac_red = np.std(red)
    ac_ir = np.std(ir)
    
    spo2 = 0
    if dc_red != 0 and dc_ir != 0 and ac_ir != 0:
        R = (ac_red / dc_red) / (ac_ir / dc_ir)
        spo2 = SPO2_A - (SPO2_B * R)

    return bpm, spo2, mean_ipi, sdnn

# --- 4. MAIN DASHBOARD APPLICATION ---

def make_document(doc):
    
    source = ColumnDataSource(data=dict(timestamp=[], red=[], ir=[]))
    
    # -- Plots --
    p_trace = figure(
        title="Real-time Pulse Oximetry Trace (Smoothed)", 
        height=400, width=900, 
        x_axis_type="datetime",
        tools="pan,box_zoom,reset,save"
    )
    p_trace.line(x='timestamp', y='red', source=source, color="#FF4136", legend_label="Red", line_width=2)
    p_trace.line(x='timestamp', y='ir', source=source, color="#B10DC9", legend_label="IR", line_width=2)
    
    p_trace.y_range.follow = "end"
    p_trace.y_range.follow_interval = 5000 
    
    # -- Metrics Display --
    style_box = {
        'background': '#f4f4f4', 'border': '1px solid #ccc', 'padding': '10px', 
        'border-radius': '5px', 'margin': '5px', 'text-align': 'center'
    }
    
    div_bpm = Div(text="<h2 style='color:#333'>--</h2><p>BPM</p>", width=200, styles=style_box)
    div_spo2 = Div(text="<h2 style='color:#0074D9'>-- %</h2><p>SpO2</p>", width=200, styles=style_box)
    div_stats = Div(text="<b>Inter-pulse Stats:</b><br>Mean IPI: -- ms<br>SDNN: -- ms", width=420, styles={'padding': '10px', 'color': '#555'})
    status_div = Div(text="<b>Status:</b> Disconnected", width=900, styles={'padding': '5px', 'color': 'gray'})

    # -- Controls --
    port_input = TextInput(value=app_state['arduino_port'] or "COM3", title="Port:")
    btn_connect = Button(label="Connect & Stream", button_type="success", width=150)
    btn_download = Button(label="Download Trace", button_type="primary", width=150)
    btn_stop = Button(label="Stop", button_type="danger", width=100)
    
    # -- Callbacks --
    def update_data():
        ser = app_state['serial_connection']
        if ser and ser.is_open:
            new_data = {'timestamp': [], 'red': [], 'ir': []}
            while ser.in_waiting:
                try:
                    line = ser.readline().decode('utf-8').strip()
                    parts = line.split(',')
                    if len(parts) == 3:
                        t, r_raw, i_raw = parts
                        
                        ts = float(t)
                        raw_red = float(r_raw)
                        raw_ir = float(i_raw)

                        # --- SMOOTHING FILTER (Exponential Moving Average) ---
                        # Algorithm: current = alpha * raw + (1 - alpha) * previous
                        
                        # Initialize if first sample
                        if app_state['last_red'] is None:
                            app_state['last_red'] = raw_red
                            app_state['last_ir'] = raw_ir
                        
                        # Apply Filter
                        smooth_red = (SMOOTHING_ALPHA * raw_red) + ((1 - SMOOTHING_ALPHA) * app_state['last_red'])
                        smooth_ir = (SMOOTHING_ALPHA * raw_ir) + ((1 - SMOOTHING_ALPHA) * app_state['last_ir'])
                        
                        # Store state for next loop
                        app_state['last_red'] = smooth_red
                        app_state['last_ir'] = smooth_ir

                        # Append SMOOTHED values to data dict
                        new_data['timestamp'].append(ts) 
                        new_data['red'].append(smooth_red)
                        new_data['ir'].append(smooth_ir)
                except: continue
            
            if new_data['timestamp']:
                source.stream(new_data, rollover=1000)
                
                # Save data (We save the smoothed data here)
                for i in range(len(new_data['timestamp'])):
                    app_state['recorded_data'].append({
                        'timestamp': new_data['timestamp'][i],
                        'red': new_data['red'][i],
                        'ir': new_data['ir'][i]
                    })
                
                # Update Metrics every ~0.5s
                if len(app_state['recorded_data']) % 20 == 0:
                    df_live = pd.DataFrame(source.data)
                    bpm, spo2, ipi, sdnn = process_signal(df_live)
                    
                    div_bpm.text = f"<h2 style='color:#333'>{bpm:.1f}</h2><p>Heart Rate (BPM)</p>"
                    div_spo2.text = f"<h2 style='color:#0074D9'>{spo2:.1f} %</h2><p>SpO2</p>"
                    div_stats.text = f"<b>Inter-pulse Stats:</b><br>Mean IPI: {ipi:.0f} ms<br>SDNN: {sdnn:.1f} ms"

    def on_connect():
        try:
            ser = serial.Serial(port_input.value, 115200, timeout=1)
            handshake_arduino(ser, print_handshake_message=True)
            app_state['serial_connection'] = ser
            app_state['streaming'] = True
            # Reset filter state on new connection
            app_state['last_red'] = None
            app_state['last_ir'] = None
            
            if app_state['periodic_callback'] is None:
                app_state['periodic_callback'] = doc.add_periodic_callback(update_data, 50)
            status_div.text = "<b>Status:</b> Connected."
            btn_connect.disabled = True
            btn_stop.disabled = False
        except Exception as e:
            status_div.text = f"<b>Status:</b> Error: {e}"

    def on_stop():
        if app_state['periodic_callback']:
            doc.remove_periodic_callback(app_state['periodic_callback'])
            app_state['periodic_callback'] = None
        if app_state['serial_connection']:
            app_state['serial_connection'].close()
        status_div.text = "<b>Status:</b> Stopped."
        btn_connect.disabled = False
        btn_stop.disabled = True

    def on_download():
        if app_state['recorded_data']:
            filename = f"pulse_ox_{int(time.time())}.csv"
            pd.DataFrame(app_state['recorded_data']).to_csv(filename, index=False)
            status_div.text = f"<b>Status:</b> Saved to {filename}"

    # -- Layout --
    btn_connect.on_click(on_connect)
    btn_stop.on_click(on_stop)
    btn_download.on_click(on_download)
    
    layout = column(
        row(port_input, btn_connect, btn_stop, btn_download),
        status_div,
        row(div_bpm, div_spo2, div_stats),
        p_trace
    )
    doc.add_root(layout)

# --- EXECUTION ---
output_notebook()


Loading BokehJS ...

In [3]:

# --- 6. RUN THE APP ---
show(make_document, notebook_url="localhost:8888")

Handshake message: 1917,1023,1023

